# espnet_onnx demonstration

This notebook provides a simple demonstration of how to export your trained model and use it for inference.

see also:
- ESPnet: https://github.com/espnet/espnet
- espnet_onnx: https://github.com/Masao-Someki/espnet_onnx

Author: [Masao Someki](https://github.com/Masao-Someki)


## Table of Contents

- Install Dependency
- Export your model
- Inference with onnx

# Install Dependency

Use a fresh Python 3.12 or 3.13 kernel (the repository CI uses Python 3.13).
In Colab, select a compatible runtime before installing dependencies.

These demos use the current repository implementation and its export requirements:
- torch >= 2.11.0 and a compatible torchaudio version
- espnet >= 202611 and espnet_model_zoo
- numpy >= 2.0.0, librosa >= 0.10.2, onnx >= 1.22.0, and onnxscript

Run the installation cell once, then restart the kernel/session before continuing.
The espnet_onnx source revision is pinned so the demo uses the exporter shown here.


In [ ]:
%pip install --upgrade "espnet_onnx @ git+https://github.com/espnet/espnet_onnx.git@a5afc78ea3e850abd1737b39e120b73295197374" "espnet>=202611" espnet_model_zoo "numpy>=2.0.0" "librosa>=0.10.2" "onnx>=1.22.0" onnxscript "protobuf>=5.29.6" scipy pyworld
%pip check

For GPU inference, select a GPU runtime with CUDA/cuDNN compatible with
[onnxruntime-gpu](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements).
The following cell replaces the CPU runtime package with the GPU package.
Restart the kernel/session after installation, before exporting or running inference.

In [ ]:
# Install only one ONNX Runtime package in the environment.
%pip uninstall -y onnxruntime onnxruntime-gpu
%pip install --upgrade onnxruntime-gpu
%pip check

# Export your model

## Export model from espnet_model_zoo

The easiest way to export a model is to use `espnet_model_zoo`. You can download, unpack, and export the pretrained models with `export_from_pretrained` method.
`espnet_onnx` will save the onnx models into cache directory, which is `${HOME}/.cache/espnet_onnx` in default.

In [ ]:
# export the model.
from espnet_onnx.export import ASRModelExport

tag_name = 'pyf98/librispeech_conformer_hop_length160'

m = ASRModelExport()
m.set_export_config(
    max_seq_len=5000,
)
m.export_from_pretrained(
    tag_name
)

# Inference with onnxruntime
Now, let's use the exported models for inference.
Please enable the GPU resource to run the following codes.

In [ ]:
# please provide the tag_name to specify exported model.
tag_name = 'pyf98/librispeech_conformer_hop_length160'

# upload wav file and let's inference!
import librosa
from google.colab import files

wav_file = files.upload()
y, sr = librosa.load(list(wav_file.keys())[0], sr=16000)

# Use the exported onnx file to inference.
from espnet_onnx import Speech2Text

speech2text = Speech2Text(tag_name, providers=['CUDAExecutionProvider'])
nbest = speech2text(y)
print(nbest[0][0])